# Predicting Departure Delays: All Models Evaluation

This notebook applies Random Forest, Lasso, Ridge, and Gradient Boosted Tree regression to the flight delay prediction task, implementing it as a regression problem with `DEP_DELAY` as the target variable. The joined OTPW-style dataset is used to construct features available prior to scheduled departure, apply a chronological train/validation/test split, tune model hyperparameters on the validation window, and compare final model performance on a held-out test set.

Data loading, feature engineering, and model training are driven by shared functions in `modeling_utils.py`, ensuring consistency across all notebooks.

The shared metrics reported in this notebook are `RMSE`, `MAE`, `R2`, `OTPA`, `SDDR`, and `F1`.

In [0]:
import sys
import importlib.util

if "modeling_utils" in sys.modules:
    del sys.modules["modeling_utils"]

sys.dont_write_bytecode = True

spec = importlib.util.spec_from_file_location(
    "modeling_utils",
    "/Workspace/Users/andrewjlei31@berkeley.edu/w_261_final_project/modeling_utils.py"
)
modeling_utils = importlib.util.module_from_spec(spec)
sys.modules["modeling_utils"] = modeling_utils
spec.loader.exec_module(modeling_utils)

In [0]:
import pandas as pd
from modeling_utils import (
    DEFAULT_DELAY_THRESHOLD,
    DEFAULT_SEVERE_DELAY_THRESHOLD,
    evaluate_model,
    fit_gbt_model,
    fit_linear_model,
    fit_rf_model,
    fit_stacking_ensemble,
    load_and_prepare_data,
    prune_empty_feature_columns,
    run_gbt_search_and_eval,
    run_linear_search_and_eval,
    search_rf_model,
    top_gbt_importances,
    top_linear_coefficients,
    top_rf_importances,
)

## Configuration

In [0]:
data_BASE_DIR = "dbfs:/mnt/mids-w261/"
DATA_PATH = f"{data_BASE_DIR}datasets_final_project_2022/parquet_airlines_data_1y/"
ROW_FILTER = "YEAR = 2019 AND QUARTER = 4"
DATE_COL = "FL_DATE"
TARGET_COL = "DEP_DELAY"

TRAIN_END = "2019-11-15"
VALID_END = "2019-11-30"
AUTO_INFER_SPLITS = False
SEED = 0

DELAY_THRESHOLD = DEFAULT_DELAY_THRESHOLD
SEVERE_DELAY_THRESHOLD = DEFAULT_SEVERE_DELAY_THRESHOLD

RF_PARAM_GRID = [
    {"numTrees": 50,  "maxDepth": 5},
    {"numTrees": 100, "maxDepth": 10},
    {"numTrees": 100, "maxDepth": 15},
    {"numTrees": 200, "maxDepth": 10},
    {"numTrees": 200, "maxDepth": 20},
    {"numTrees": 200, "maxDepth": 30},
]

LASSO_REG_PARAMS = [0.001, 0.01, 0.05]
RIDGE_REG_PARAMS = [0.01, 0.10, 1.00]
LINEAR_MAX_ITER = 100

GBT_PARAM_GRID = [
    {"maxDepth": 5, "maxIter": 40, "stepSize": 0.05},
    {"maxDepth": 7, "maxIter": 60, "stepSize": 0.05},
    {"maxDepth": 5, "maxIter": 80, "stepSize": 0.10},
]

In [0]:
# start Spark Session (RUN THIS CELL AS IS)
#Step A: Start Spark by running the following cell

import os
import sys
os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@17'

import pyspark
from pyspark.sql import SparkSession

# Clean up stale 'spark' module if it was imported by mistake earlier
if 'spark' in sys.modules and not isinstance(sys.modules['spark'], SparkSession):
    del sys.modules['spark']
    try:
        del spark
    except NameError:
        pass

try:
    spark
    print('Spark is already running')
    sc = spark.sparkContext
    print(f'{sc.master} appName: {sc.appName}')
except NameError:
    print('starting Spark')
    app_name = 'random_forest'
    master = 'local[*]'
    spark = SparkSession\
            .builder\
            .appName(app_name)\
            .master(master)\
            .getOrCreate()

# Don't worry about messages shown below

Spark is already running
spark://10.59.221.177:7077 appName: Databricks Shell


## Data Ingestion and Feature Engineering

Data is loaded from the OTPW joined dataset, feature-engineered (weather flags, scheduled departure hour, weekend indicator, numeric parsing), and split chronologically into train/validation/test sets — all via `load_and_prepare_data()` from `modeling_utils`.

In [0]:
data = load_and_prepare_data(
    spark,
    data_path=DATA_PATH,
    row_filter=ROW_FILTER,
    target_col=TARGET_COL,
    date_col=DATE_COL,
    train_end=TRAIN_END,
    valid_end=VALID_END,
    auto_infer_splits=AUTO_INFER_SPLITS,
    seed=SEED,
)

before_count = train_df.count() + valid_df.count() + test_df.count()
train_df = train_df.dropna()
valid_df = valid_df.dropna()
test_df = test_df.dropna()
after_count = train_df.count() + valid_df.count() + test_df.count()

print(before_count)
print(after_count)

model_df = data["model_df"]
train_df = data["train_df"]
valid_df = data["valid_df"]
test_df = data["test_df"]
split_summary = data["split_summary"]
numeric_cols = data["numeric_cols"]
linear_cat_cols = data["linear_cat_cols"]
tree_cat_cols = data["tree_cat_cols"]

print("Split summary")
print(split_summary.to_string(index=False))
print("\nNumeric features:", numeric_cols)
print("Linear categorical features:", linear_cat_cols)
print("Tree categorical features:", tree_cat_cols)

split_summary

3690712
3690712
Split summary
     split    rows   min_date   max_date
     train 1866432 2019-10-01 2019-11-15
validation  587056 2019-11-16 2019-11-30
      test 1237224 2019-12-01 2019-12-31

Numeric features: ['DISTANCE', 'DAY_OF_WEEK', 'MONTH', 'QUARTER', 'scheduled_dep_hour', 'is_weekend', 'has_rain', 'has_snow', 'has_fog', 'has_thunder']
Linear categorical features: ['OP_CARRIER', 'ORIGIN', 'DEST', 'DEP_TIME_BLK', 'ORIGIN_STATE_ABR', 'DEST_STATE_ABR']
Tree categorical features: ['OP_CARRIER', 'DEP_TIME_BLK']


,split,rows,min_date,max_date
0,train,1866432,2019-10-01,2019-11-15
1,validation,587056,2019-11-16,2019-11-30
2,test,1237224,2019-12-01,2019-12-31


## Lasso and Ridge Training

Lasso and Ridge regularized linear models are searched and evaluated via `run_linear_search_and_eval()` from `modeling_utils`, using the same data prepared above.

In [0]:
linear_results = run_linear_search_and_eval(
    train_df=train_df,
    valid_df=valid_df,
    test_df=test_df,
    numeric_cols=numeric_cols,
    linear_cat_cols=linear_cat_cols,
    lasso_reg_params=LASSO_REG_PARAMS,
    ridge_reg_params=RIDGE_REG_PARAMS,
    linear_max_iter=LINEAR_MAX_ITER,
    delay_threshold=DELAY_THRESHOLD,
    severe_delay_threshold=SEVERE_DELAY_THRESHOLD,
)

print("Lasso validation search:")
display(linear_results["lasso_search"]) if "display" in dir() else print(linear_results["lasso_search"])
print("\nRidge validation search:")
display(linear_results["ridge_search"]) if "display" in dir() else print(linear_results["ridge_search"])

Lasso validation search:


model,split,regParam,elasticNetParam,maxIter,RMSE,MAE,R2,OTPA,SDDR,F1,rows
Lasso,validation,0.01,1.0,100,40.35923673241796,16.2297002555634,0.004767803555480454,0.8468152953040256,0.0,0.10891795481569558,587056
Lasso,validation,0.05,1.0,100,40.36847029647192,16.33084100610475,0.004312364246401135,0.8613113570085307,0.0,0.043379156385853594,587056
Lasso,validation,0.001,1.0,100,40.370024376462815,16.22363165474436,0.004235700057466318,0.8380835899811943,0.0,0.1329246711546531,587056



Ridge validation search:


model,split,regParam,elasticNetParam,maxIter,RMSE,MAE,R2,OTPA,SDDR,F1,rows
Ridge,validation,1.0,0.0,100,40.351631911407004,16.254650402284586,0.005142827974445763,0.8624492382328092,0.0,0.0386675873235077,587056
Ridge,validation,0.1,0.0,100,40.359564840738855,16.218382706006725,0.004751621619514257,0.8448154860865063,0.0,0.11508499271491014,587056
Ridge,validation,0.01,0.0,100,40.37299654152772,16.22201830218304,0.0040890722137924,0.836996811207108,0.0,0.13497975122938966,587056


## Gradient Boosted Trees Training

GBT is searched and evaluated via `run_gbt_search_and_eval()` from `modeling_utils`, reusing the `train_valid_df` already computed by the linear step.

In [0]:
gbt_results = run_gbt_search_and_eval(
    train_df=train_df,
    valid_df=valid_df,
    test_df=test_df,
    numeric_cols=numeric_cols,
    tree_cat_cols=tree_cat_cols,
    gbt_param_grid=GBT_PARAM_GRID,
    delay_threshold=DELAY_THRESHOLD,
    severe_delay_threshold=SEVERE_DELAY_THRESHOLD,
    seed=SEED,
    train_valid_df=linear_results["train_valid_df"],
)

print("GBT validation search:")
display(gbt_results["gbt_search"]) if "display" in dir() else print(gbt_results["gbt_search"])

GBT validation search:


model,split,maxDepth,maxIter,stepSize,RMSE,MAE,R2,OTPA,SDDR,F1,rows
GBT,validation,5,40,0.05,40.55424870315159,16.472002988686306,-0.0048731659478811196,0.8132818674879398,0.0,0.12034347163149026,587056
GBT,validation,5,80,0.1,40.630664843767626,16.510772145071027,-0.008663687419885102,0.8096570003543103,3.3647375504710633E-4,0.12434762165974453,587056
GBT,validation,7,60,0.05,40.892608760073344,16.51970886921403,-0.021711221971709938,0.8091868578125426,8.411843876177658E-4,0.1227347482183413,587056


## Random Forest Training and Validation

Random Forest is trained on the same data splits. Hyperparameters are searched on the validation set, then the best configuration is refit on train+validation for final test evaluation.

In [0]:
rf_numeric_cols, rf_cat_cols = prune_empty_feature_columns(train_df, numeric_cols, tree_cat_cols)

best_rf, rf_search = search_rf_model(
    train_df=train_df,
    valid_df=valid_df,
    numeric_columns=rf_numeric_cols,
    categorical_columns=rf_cat_cols,
    param_grid=RF_PARAM_GRID,
    model_name="RandomForest",
    delay_threshold=DELAY_THRESHOLD,
    severe_delay_threshold=SEVERE_DELAY_THRESHOLD,
    seed=SEED,
)

print("RF validation search results:")
rf_search

RF validation search results:


,model,split,numTrees,maxDepth,maxBins,subsamplingRate,RMSE,MAE,R2,OTPA,SDDR,F1,rows
0,RandomForest,validation,100,10,32,0.8,40.411085,16.528040,0.002209,0.868711,0.0,None,587056
1,RandomForest,validation,100,15,32,0.8,40.411085,16.528040,0.002209,0.868711,0.0,None,587056
2,RandomForest,validation,200,10,32,0.8,40.413088,16.537858,0.002110,0.868711,0.0,None,587056
3,RandomForest,validation,200,20,32,0.8,40.413088,16.537858,0.002110,0.868711,0.0,None,587056
4,RandomForest,validation,200,30,32,0.8,40.413088,16.537858,0.002110,0.868711,0.0,None,587056
5,RandomForest,validation,50,5,32,0.8,40.424124,16.517668,0.001565,0.868704,0.0,None,587056


## Final Model Evaluation

After selecting the best validation configuration for each model, the training and validation windows are combined and the Random Forest is refit on the enlarged development set. Final performance for all four models (Lasso, Ridge, GBT, Random Forest) is compared on the held-out test set.

In [0]:
train_valid_df = linear_results["train_valid_df"]

rf_model = fit_rf_model(
    train_df=train_valid_df,
    numeric_columns=rf_numeric_cols,
    categorical_columns=rf_cat_cols,
    num_trees=int(best_rf["numTrees"]),
    max_depth=int(best_rf["maxDepth"]),
    max_bins=int(best_rf["maxBins"]),
    subsampling_rate=float(best_rf["subsamplingRate"]),
    seed=SEED,
)

test_results = pd.DataFrame(
    [
        linear_results["lasso_test"],
        linear_results["ridge_test"],
        gbt_results["gbt_test"],
        {
            **evaluate_model(rf_model, test_df, model_name="RandomForest", split_name="test",
                             delay_threshold=DELAY_THRESHOLD, severe_delay_threshold=SEVERE_DELAY_THRESHOLD),
            "numTrees": best_rf["numTrees"],
            "maxDepth": best_rf["maxDepth"],
            "maxBins": best_rf["maxBins"],
            "subsamplingRate": best_rf["subsamplingRate"],
        },
    ]
).sort_values(["RMSE", "model"]).reset_index(drop=True)

test_results

,model,split,RMSE,MAE,R2,OTPA,SDDR,F1,rows,regParam,elasticNetParam,maxIter,maxDepth,stepSize,numTrees,maxBins,subsamplingRate
0,GBT,test,50.904953,19.637332,-0.002153,0.798795,0.000024,0.022784,1237224,NaN,NaN,40.0,5.0,0.05,NaN,NaN,NaN
1,RandomForest,test,50.943765,19.923790,-0.003681,0.800865,0.000000,NaN,1237224,NaN,NaN,NaN,10.0,NaN,100.0,32.0,0.8
2,Lasso,test,51.046828,19.105626,-0.007747,0.798284,0.000000,0.042891,1237224,0.01,1.0,100.0,NaN,NaN,NaN,NaN,NaN
3,Ridge,test,51.100240,19.040213,-0.009857,0.800772,0.000000,0.002509,1237224,1.00,0.0,100.0,NaN,NaN,NaN,NaN,NaN


## Model Interpretation

Top learned signals for each model: largest absolute coefficients for Lasso/Ridge, ranked feature importances for GBT and Random Forest.

In [0]:
lasso_features = top_linear_coefficients(linear_results["lasso_model"], train_valid_df, top_n=20)
ridge_features = top_linear_coefficients(linear_results["ridge_model"], train_valid_df, top_n=20)
gbt_features = top_gbt_importances(gbt_results["gbt_model"], train_valid_df, top_n=20)
rf_features = top_rf_importances(rf_model, train_valid_df, top_n=20)

for name, table in [("Lasso", lasso_features), ("Ridge", ridge_features),
                     ("GBT", gbt_features), ("Random Forest", rf_features)]:
    print(f"\nTop signals for {name}\n")
    if "display" in globals():
        display(table)
    else:
        print(table.to_string(index=False))


Top signals for Lasso



feature,coefficient,abs_coefficient
DEST_ohe_EWR,10.470819204542146,10.470819204542146
DEP_TIME_BLK_ohe_2300-2359,-6.279248019171283,6.279248019171283
DEST_ohe_LGA,5.832221467162028,5.832221467162028
ORIGIN_ohe_EWR,5.500649761150069,5.500649761150069
OP_CARRIER_ohe_YX,-4.319593598291598,4.319593598291598
OP_CARRIER_ohe_AS,-4.169169897724274,4.169169897724274
OP_CARRIER_ohe_F9,3.861597392386026,3.861597392386026
OP_CARRIER_ohe_YV,3.858154116961259,3.858154116961259
OP_CARRIER_ohe_MQ,-3.7318900036187195,3.7318900036187195
DEP_TIME_BLK_ohe_2200-2259,-3.70314206852842,3.70314206852842



Top signals for Ridge



feature,coefficient,abs_coefficient
DEST_ohe_EWR,3.0927972716364414,3.0927972716364414
DEST_STATE_ABR_ohe_NJ,2.913234497740392,2.913234497740392
OP_CARRIER_ohe_YV,2.688786777501233,2.688786777501233
DEP_TIME_BLK_ohe_0600-0659,-2.404208217911778,2.404208217911778
OP_CARRIER_ohe_AS,-2.378335327966724,2.378335327966724
DEST_ohe_LGA,2.2994212020685607,2.2994212020685607
OP_CARRIER_ohe_YX,-2.251379773144946,2.251379773144946
OP_CARRIER_ohe_MQ,-2.138272426109258,2.138272426109258
OP_CARRIER_ohe_F9,2.121133670831341,2.121133670831341
OP_CARRIER_ohe_OO,2.007086762230594,2.007086762230594



Top signals for GBT



feature,importance
scheduled_dep_hour_imp,0.17628898569335255
DISTANCE_imp,0.152894610116273
DAY_OF_WEEK_imp,0.14409259365151666
MONTH_imp,0.0736410534556736
OP_CARRIER_ohe_YV,0.03838123376038681
OP_CARRIER_ohe_HA,0.032201725286527635
OP_CARRIER_ohe_AA,0.02938839096686163
OP_CARRIER_ohe_B6,0.027591478796829525
OP_CARRIER_ohe_AS,0.026967159969211847
OP_CARRIER_ohe_F9,0.02683268496480303



Top signals for Random Forest



feature,importance
scheduled_dep_hour_imp,0.4717545871528074
OP_CARRIER_ohe_DL,0.08863172527171738
DEP_TIME_BLK_ohe_0600-0659,0.0784479002110641
DAY_OF_WEEK_imp,0.07491603267560615
DEP_TIME_BLK_ohe_0700-0759,0.06201758225318149
OP_CARRIER_ohe_OO,0.04085113906151621
OP_CARRIER_ohe_AS,0.026084519629184757
is_weekend_imp,0.024785292443170997
MONTH_imp,0.022561097968996975
DISTANCE_imp,0.019894747595115923


## Stacking Ensemble

A Ridge-regularized linear regression meta-learner is trained on the validation-set predictions of all four base models. The base models used for validation predictions are trained on `train_df` only, so the meta-learner sees out-of-sample predictions. For final test evaluation, the base models retrained on `train+valid` (already fitted above) generate test predictions, which the meta-learner combines into a single stacked prediction.

In [0]:
# Refit base models on train_df only (out-of-sample for validation predictions)
lasso_train_only = fit_linear_model(
    train_df=train_df,
    numeric_columns=linear_results["linear_numeric_cols"],
    categorical_columns=linear_results["linear_cat_cols"],
    reg_param=linear_results["best_lasso"]["regParam"],
    elastic_net_param=linear_results["best_lasso"]["elasticNetParam"],
    max_iter=int(linear_results["best_lasso"]["maxIter"]),
)

ridge_train_only = fit_linear_model(
    train_df=train_df,
    numeric_columns=linear_results["linear_numeric_cols"],
    categorical_columns=linear_results["linear_cat_cols"],
    reg_param=linear_results["best_ridge"]["regParam"],
    elastic_net_param=linear_results["best_ridge"]["elasticNetParam"],
    max_iter=int(linear_results["best_ridge"]["maxIter"]),
)

gbt_train_only = fit_gbt_model(
    train_df=train_df,
    numeric_columns=gbt_results["gbt_numeric_cols"],
    categorical_columns=gbt_results["gbt_cat_cols"],
    max_depth=int(gbt_results["best_gbt"]["maxDepth"]),
    max_iter=int(gbt_results["best_gbt"]["maxIter"]),
    step_size=float(gbt_results["best_gbt"]["stepSize"]),
    seed=SEED,
)

rf_train_only = fit_rf_model(
    train_df=train_df,
    numeric_columns=rf_numeric_cols,
    categorical_columns=rf_cat_cols,
    num_trees=int(best_rf["numTrees"]),
    max_depth=int(best_rf["maxDepth"]),
    max_bins=int(best_rf["maxBins"]),
    subsampling_rate=float(best_rf["subsamplingRate"]),
    seed=SEED,
)


In [0]:
base_models_train = {
    "Lasso": lasso_train_only,
    "Ridge": ridge_train_only,
    "GBT": gbt_train_only,
    "RF": rf_train_only,
}

base_models_final = {
    "Lasso": linear_results["lasso_model"],
    "Ridge": linear_results["ridge_model"],
    "GBT": gbt_results["gbt_model"],
    "RF": rf_model,
}

stacking = fit_stacking_ensemble(
    base_models_train=base_models_train,
    base_models_final=base_models_final,
    valid_df=valid_df,
    test_df=test_df,
    meta_reg_param=0.01,
    delay_threshold=DELAY_THRESHOLD,
    severe_delay_threshold=SEVERE_DELAY_THRESHOLD,
)

print("Meta-learner weights (Ridge regression on base model predictions):")
print(stacking["meta_weights"].to_string(index=False))
print(f"\nMeta-learner intercept: {stacking['meta_intercept']:.4f}")

Meta-learner weights (Ridge regression on base model predictions):
base_model    weight
Lasso_pred  0.939630
  GBT_pred  0.073727
   RF_pred  0.004443
Ridge_pred -0.364545

Meta-learner intercept: 1.3966


In [0]:
train_results = pd.concat(
    pd.DataFrame([stacking["stacked_train_metrics"]]),
    ignore_index=True,
    sort=False,
).sort_values(["RMSE", "model"]).reset_index(drop=True)


---------------------------------------------------------------------------
KeyError                                  Traceback (most recent call last)
File <command-7866244608656029>, line 2
      1 train_results = pd.concat(
----> 2     pd.DataFrame([stacking["stacked_train_metrics"]]),
      3     ignore_index=True,
      4     sort=False,
      5 ).sort_values(["RMSE", "model"]).reset_index(drop=True)

KeyError: 'stacked_train_metrics'

In [0]:
train_results = pd.concat(
    [train_resul;ts, pd.DataFrame([stacking["stacked_test_metrics"]])],
    ignore_index=True,
    sort=False,
).sort_values(["RMSE", "model"]).reset_index(drop=True)


## Combined Test Results (All Models + Stacking)

In [0]:
all_test_results = pd.concat(
    [test_results, pd.DataFrame([stacking["stacked_test_metrics"]])],
    ignore_index=True,
    sort=False,
).sort_values(["RMSE", "model"]).reset_index(drop=True)

all_test_results

,model,split,RMSE,MAE,R2,OTPA,SDDR,F1,rows,regParam,elasticNetParam,maxIter,maxDepth,stepSize,numTrees,maxBins,subsamplingRate
0,GBT,test,50.904953,19.637332,-0.002153,0.798795,0.000024,0.022784,1237224,NaN,NaN,40.0,5.0,0.05,NaN,NaN,NaN
1,RandomForest,test,50.943765,19.923790,-0.003681,0.800865,0.000000,NaN,1237224,NaN,NaN,NaN,10.0,NaN,100.0,32.0,0.8
2,Lasso,test,51.046828,19.105626,-0.007747,0.798284,0.000000,0.042891,1237224,0.01,1.0,100.0,NaN,NaN,NaN,NaN,NaN
3,Ridge,test,51.100240,19.040213,-0.009857,0.800772,0.000000,0.002509,1237224,1.00,0.0,100.0,NaN,NaN,NaN,NaN,NaN
4,StackingEnsemble,test,51.124438,19.038469,-0.010813,0.800783,0.000000,0.004266,1237224,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
